In [0]:
describe workspace.redfin_bronze.redfin_user_growth

In [0]:
select distinct business_market, environment, device, medium
from workspace.redfin_bronze.redfin_user_growth
-- > no null business market, device, and medium 

In [0]:
select  count(*) as total_records,
        min(day) as min_day,
        max(day) as max_day,
        count(distinct day) as distinct_days,
        min(sessions) as min_session , 
        max(sessions) as max_session, 
        sum(sessions) as total_sessions
from workspace.redfin_bronze.redfin_user_growth

In [0]:
select distinct month(day) as mon , 
       sum(case when year(day)= 2018 then 1 else 0 end ) as y2018,  
       sum(case when year(day)= 2019 then 1 else 0 end ) as y2019
from workspace.redfin_bronze.redfin_user_growth
group by 1 order by 1 

Databricks visualization. Run in Databricks to view.

1. data check - day 

In [0]:
-- daily count (365 days + 211 days > 576 days )
select  day,
        count(*) as record_count
from workspace.redfin_bronze.redfin_user_growth
group by 1 order by 1 
-- > results  16 missing days 

In [0]:
select  date_trunc('month',day ) as mon, 
        count(*) as record_count
from workspace.redfin_bronze.redfin_user_growth
group by 1 order by 1 
-- > visuals shows that missing days fall in Sep 2018 

Databricks visualization. Run in Databricks to view.

In [0]:
with calendar as 
(select explode(
        sequence(
            date'2018-01-01',
            date'2019-07-30',
            interval 1 day)
    ) as calendar_day
)

select c.calendar_day as missing_day
from calendar c
left anti join ( select distinct day from workspace.redfin_bronze.redfin_user_growth) d
    on c.calendar_day = d.day
order by 1

In [0]:
select  year(day) as year,
        month(day) as month,
        count(*) as total_records,
        count(distinct environment) as environments,
        count(distinct device) as devices,
        count(distinct medium) as mediums,
        count(distinct business_market) as markets
from workspace.redfin_bronze.redfin_user_growth
group by 1,2  order by 1,2 
--> medium: up to 8 in 2018 ; up to 9 in 2019 
--> device : same 3 in 2018/19
--> environment: same 3 in 2018/19 
-- market: same 13 in 2018/19

In [0]:
--medium coverage 
select year(day) as year,
    month(day) as month,
    coalesce(nullif(trim(medium), ''), '<missing>') as medium,
    count(*) as record_count,
    sum(sessions) as total_sessions
from workspace.redfin_bronze.redfin_user_growth
group by    1,2,3 
order by 1,2 ,3 

Databricks visualization. Run in Databricks to view.

In [0]:
select
    medium,
    count(*) as record_count,
    count(distinct day) as observed_days,
    min(day) as first_day,
    max(day) as last_day,
    sum(sessions) as total_sessions
from workspace.redfin_bronze.redfin_user_growth
where lower(trim(medium)) in ('social', 'share', 'internal')
group by 1 order by 1 

In [0]:
---- overall traffic trend 
select year(day) as year,
        month(day) as month,
        count(distinct day) as observed_days,
        sum(sessions) as total_sessions,
        round(sum(sessions) / count(distinct day), 0) as avg_daily_sessions
from workspace.redfin_bronze.redfin_user_growth
group by 1,2  order by 1,2 

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
with session as 
(select year(day) as year,
        month(day) as month,
        count(distinct day) as observed_days,
        sum(sessions) as total_sessions,
        round(sum(sessions) / count(distinct day), 0) as avg_daily_sessions
from workspace.redfin_bronze.redfin_user_growth
group by 1,2  order by 1,2 )

select  distinct month, 
        sum(case when year = 2018 then avg_daily_sessions else null   end)as y2018 , 
        sum(case when year = 2019 then avg_daily_sessions else null  end)  as y2019 , 
        round(
            (sum(case when year = 2019 then avg_daily_sessions else null end)*1.0/sum(case when year = 2018 then avg_daily_sessions else null end))-1, 2)  as yoy 
from session 
group by 1 


In [0]:
-- where are the data from 
select  year(day) as year,
        month(day) as month,
        environment,
        sum(sessions) as total_sessions,
        round(sum(sessions) / count(distinct day), 0) as avg_daily_sessions
from workspace.redfin_bronze.redfin_user_growth
group by  1,2,3 
order by 1,2,3 

In [0]:
--device 
select    year(day) as year,
    month(day) as month,
    device,
    sum(sessions) as total_sessions,
    round(sum(sessions) / count(distinct day), 0) as avg_daily_sessions
from workspace.redfin_bronze.redfin_user_growth
group by 1,2,3 
order by 1,2,3 